In [1]:
# Semente fixa para reprodutibilidade
RANDOM_STATE = 42

# 02 — Pré-processamento e Engenharia de Atributos

## Grupo 50 • Tech Challenge — Fase 2

Este notebook integra as bases, constrói o `TARGET`, identifica perfis cadastrais repetidos e cria atributos derivados para a modelagem.

**Princípio metodológico:** o `PROFILE_GROUP` é criado a partir das características cadastrais originais, sem `ID` e sem `TARGET`, antes das transformações derivadas. Assim, a definição do perfil não depende de estatísticas calculadas durante o pré-processamento.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Localização robusta para execução no repositório local ou no Colab.
candidatos = [
    Path.cwd().parent / "data" / "raw",
    Path.cwd() / "data" / "raw",
    Path("/content/data/raw"),
]
RAW = next((p for p in candidatos if (p / "application_record.csv").exists()), None)

if RAW is None and Path("/content/application_record.csv").exists():
    RAW = Path("/content")

if RAW is None:
    raise FileNotFoundError("Não foi possível localizar application_record.csv e credit_record.csv.")

ROOT = RAW.parent.parent if RAW.name == "raw" else RAW
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

app = pd.read_csv(RAW / "application_record.csv")
credit = pd.read_csv(RAW / "credit_record.csv")

print("Application:", app.shape)
print("Credit:", credit.shape)

Application: (438557, 18)
Credit: (1048575, 3)


## 1. Construção do TARGET

Clientes que apresentaram pelo menos um registro nos status `2`, `3`, `4` ou `5` são classificados como maus pagadores (`TARGET = 1`). Os demais ficam como bons pagadores (`TARGET = 0`).

O histórico de crédito é utilizado apenas para construir a variável-alvo.

In [3]:
application_clean = app.drop_duplicates("ID", keep="first").copy()

credit["STATUS_NUM"] = pd.to_numeric(credit["STATUS"], errors="coerce")
worst_status = credit.groupby("ID")["STATUS_NUM"].max().rename("WORST_STATUS")
target = worst_status.isin([2, 3, 4, 5]).astype(int).rename("TARGET")

df_model = application_clean.merge(
    target,
    left_on="ID",
    right_index=True,
    how="inner",
)

print("Aplicações originais:", len(app))
print("Aplicações únicas por ID:", len(application_clean))
print("Base final:", df_model.shape)
print()
print(df_model["TARGET"].value_counts().rename(index={0: "Bom pagador", 1: "Mau pagador"}))
print()
print((df_model["TARGET"].value_counts(normalize=True) * 100).round(2))

Aplicações originais: 438557
Aplicações únicas por ID: 438510
Base final: (36457, 19)

TARGET
Bom pagador    35841
Mau pagador      616
Name: count, dtype: int64

TARGET
0    98.31
1     1.69
Name: proportion, dtype: float64


## 2. Identificação dos perfis cadastrais

O perfil é definido pelas características cadastrais originais da aplicação, excluindo `ID` e `TARGET`.

Essa identificação é feita **antes** da engenharia de atributos e do tratamento de valores ausentes. Dessa forma, dois registros somente pertencem ao mesmo grupo quando o cadastro original é equivalente.

In [4]:
colunas_perfil = [c for c in application_clean.columns if c != "ID"]

profile_key = (
    df_model[colunas_perfil]
    .astype(object)
    .where(df_model[colunas_perfil].notna(), "<NA>")
    .astype(str)
    .agg("||".join, axis=1)
)

df_model["PROFILE_GROUP"] = pd.factorize(profile_key, sort=True)[0]

conflitos = df_model.groupby("PROFILE_GROUP")["TARGET"].nunique()

print("Perfis cadastrais distintos:", df_model["PROFILE_GROUP"].nunique())
print("Clientes em perfis repetidos:", int(df_model["PROFILE_GROUP"].duplicated(keep=False).sum()))
print("Perfis com TARGET conflitante:", int((conflitos > 1).sum()))
print("Maior tamanho de grupo:", int(df_model["PROFILE_GROUP"].value_counts().max()))

Perfis cadastrais distintos: 9728
Clientes em perfis repetidos: 33320
Perfis com TARGET conflitante: 269
Maior tamanho de grupo: 51


## 3. Engenharia de atributos

São criados:

- `AGE_YEARS`: idade aproximada em anos;
- `EMPLOYED_YEARS`: tempo de emprego aproximado em anos.

O valor `365243` em `DAYS_EMPLOYED` é tratado como informação especial/ausente.

As variáveis brutas permanecem na base processada para que a etapa de modelagem possa reproduzir explicitamente quais campos entram e quais são excluídos.

In [5]:
df_model["AGE_YEARS"] = (-df_model["DAYS_BIRTH"] / 365.25).round(2)

df_model["EMPLOYED_YEARS"] = (
    df_model["DAYS_EMPLOYED"]
    .where(df_model["DAYS_EMPLOYED"] < 365243, np.nan)
    .abs()
    .div(365.25)
    .clip(upper=60)
    .round(2)
)

print("Atributos derivados criados.")
display(df_model[["AGE_YEARS", "EMPLOYED_YEARS"]].describe())

Atributos derivados criados.


,AGE_YEARS,EMPLOYED_YEARS
count,36457.000000,30322.000000
mean,43.737717,7.243014
std,11.500502,6.454193
min,20.500000,0.050000
25%,34.120000,2.680000
50%,42.610000,5.450000
75%,53.220000,9.600000
max,68.860000,43.020000


## 4. Tratamento inicial dos tipos

Valores categóricos não são preenchidos neste notebook para que a etapa de modelagem possa aprender os parâmetros a partir do conjunto de treino.

A base processada mantém os dados necessários para a divisão agrupada e para a transformação posterior.

In [6]:
print("Valores ausentes por coluna — principais ocorrências:")
display(
    df_model.isna().mean()
    .sort_values(ascending=False)
    .head(10)
    .mul(100)
    .round(2)
    .to_frame("percentual_nulo")
)

Valores ausentes por coluna — principais ocorrências:


,percentual_nulo
OCCUPATION_TYPE,31.06
EMPLOYED_YEARS,16.83
FLAG_OWN_CAR,0.00
FLAG_OWN_REALTY,0.00
ID,0.00
CODE_GENDER,0.00
AMT_INCOME_TOTAL,0.00
CNT_CHILDREN,0.00
NAME_INCOME_TYPE,0.00
NAME_EDUCATION_TYPE,0.00


## 5. Verificações finais

`PROFILE_GROUP` é utilizado exclusivamente para impedir sobreposição de perfis entre treino, validação e teste. Ele não é utilizado como variável preditora.

In [7]:
print("TARGET nulos:", int(df_model["TARGET"].isna().sum()))
print("PROFILE_GROUP nulos:", int(df_model["PROFILE_GROUP"].isna().sum()))
print("Linhas da base final:", len(df_model))
print("Colunas da base final:", df_model.shape[1])

output_path = PROC / "base_modelagem.csv"
df_model.to_csv(output_path, index=False)

print("Base de modelagem salva em:", output_path)
print("Shape salvo:", df_model.shape)

TARGET nulos: 0
PROFILE_GROUP nulos: 0
Linhas da base final: 36457
Colunas da base final: 22
Base de modelagem salva em: c:\projetos\Grupo_50_Tech-Challenge-Fase-2\data\processed\base_modelagem.csv
Shape salvo: (36457, 22)


## Conclusão

A base de modelagem mantém o cadastro original, o `TARGET`, os atributos derivados e o `PROFILE_GROUP`. A divisão agrupada e as transformações aprendidas a partir do treino ficam para o notebook de modelagem.